#Powered by [@CoinNoin](https://www.youtube.com/@CoinNoin)
[![Subscribe](https://img.shields.io/badge/YouTube-Subscribe%20@CoinNoin-red?style=for-the-badge&logo=youtube)](https://www.youtube.com/@CoinNoin)

In [ ]:
#@title 1. Initialize Core Environment
#@markdown This prepares the ephemeral storage, installs ComfyUI, and configures the GGUF integration tools.

import os
import subprocess
from IPython.display import clear_output

LOCAL_WORKSPACE = "/content/ComfyUI"

print("🚀 [@CoinNoin] Initializing Core Architecture...")
if not os.path.exists(LOCAL_WORKSPACE):
    !git clone https://github.com/comfyanonymous/ComfyUI {LOCAL_WORKSPACE} &> /dev/null
    print("   ✓ Core Engine Cloned")
else:
    !cd {LOCAL_WORKSPACE} && git pull &> /dev/null
    print("   ✓ Core Engine Updated")

print("📦 [@CoinNoin] Installing Dependencies (This takes a moment)...")
!cd {LOCAL_WORKSPACE} && pip install xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121 &> /dev/null

GGUF_NODE_DIR = os.path.join(LOCAL_WORKSPACE, "custom_nodes/ComfyUI-GGUF")
if not os.path.exists(GGUF_NODE_DIR):
    print("🧩 [@CoinNoin] Installing GGUF Processing Nodes...")
    !git clone https://github.com/city96/ComfyUI-GGUF {GGUF_NODE_DIR} &> /dev/null
    !pip install -r {GGUF_NODE_DIR}/requirements.txt &> /dev/null
clear_output()
print("✅ [@CoinNoin] Environment Ready!")

In [ ]:
#@title 2. High-Speed Asset Downloader & LoRA Setup
#@markdown The fixed **Z-Image Turbo Q8_0 GGUF** and required base assets are fetched automatically. You can optionally download or upload a LoRA file.

# --- Optional LoRA Source ---
LORA_SOURCE = "None (No LoRA)" #@param ["None (No LoRA)", "Download from URL", "Upload from Computer"]
LORA_URLS = "" #@param {type:"string"}

import os
import subprocess
import urllib.parse
from google.colab import files

WORKSPACE = "/content/ComfyUI"

# Fixed UNet model (Locked to Z-Image Turbo Q8_0 GGUF)
FIXED_UNET_URL = "https://huggingface.co/unsloth/Z-Image-Turbo-GGUF/resolve/main/z-image-turbo-Q8_0.gguf"
TEXT_ENCODER_URLS = "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors"
VAE_URLS = "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors"

DIRS = {
    "unet":  os.path.join(WORKSPACE, "models/unet"),
    "clip":  os.path.join(WORKSPACE, "models/clip"),
    "vae":   os.path.join(WORKSPACE, "models/vae"),
    "loras": os.path.join(WORKSPACE, "models/loras"),
}

for p in DIRS.values():
    os.makedirs(p, exist_ok=True)

print("⚡ [@CoinNoin] Configuring Aria2c Accelerator...")
subprocess.run(['apt-get', '-y', 'install', '-qq', 'aria2'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def download_file(url, target_dir):
    if not url.strip(): return
    parsed_url = urllib.parse.urlparse(url)
    filename = os.path.basename(parsed_url.path)
    print(f"   📥 Fetching: {filename if filename else url[:40]}...")

    aria2_cmd = [
        "aria2c", "--console-log-level=error", "--summary-interval=10",
        "-c", "-x", "16", "-s", "16", "-k", "1M"
    ]
    if "civitai.com" in url or not filename:
        aria2_cmd.extend(["--content-disposition", url, "-d", target_dir])
    else:
        aria2_cmd.extend(["-o", filename, url, "-d", target_dir])

    subprocess.run(aria2_cmd, check=True)
    print(f"   ✅ Saved in {os.path.basename(target_dir)}")

# Download required base assets
download_file(FIXED_UNET_URL, DIRS["unet"])
download_file(TEXT_ENCODER_URLS, DIRS["clip"])
download_file(VAE_URLS, DIRS["vae"])

# Handle optional LoRA
ACTIVE_LORA_FILENAME = ""

if LORA_SOURCE == "Download from URL":
    if LORA_URLS.strip():
        download_file(LORA_URLS.strip(), DIRS["loras"])
        parsed_name = os.path.basename(urllib.parse.urlparse(LORA_URLS.strip()).path)
        ACTIVE_LORA_FILENAME = parsed_name
    else:
        print("⚠️ No LoRA URL provided. Proceeding without LoRA.")
elif LORA_SOURCE == "Upload from Computer":
    print("\n📤 [@CoinNoin] Select your LoRA (.safetensors) file to upload:")
    uploaded = files.upload()
    if uploaded:
        for fname in uploaded.keys():
            dest = os.path.join(DIRS["loras"], fname)
            os.replace(fname, dest)
            ACTIVE_LORA_FILENAME = fname
            print(f"   ✅ LoRA uploaded: {fname}")
    else:
        print("⚠️ No file uploaded. Proceeding without LoRA.")
else:
    print("ℹ️ Running in Pure Base Model Mode (No LoRA).")

# List all available LoRAs in the directory
available_loras = [f for f in os.listdir(DIRS["loras"]) if f.endswith(('.safetensors', '.pt', '.ckpt'))]
print("\n📂 [@CoinNoin] Available LoRAs in storage:")
if available_loras:
    for idx, lora_name in enumerate(available_loras, 1):
        print(f"   {idx}. {lora_name}")
        if idx == 1:
          recent_lora = {lora_name}

else:
    print("   (None found - model will run without LoRA)")

print("\n✅ [@CoinNoin] All core assets secured!")

In [ ]:
#@title 3. Image Generation
#@markdown Enter your prompt, tweak parameters, or select your LoRA from the dropdown.

# --- Model & LoRA Selection ---
#@markdown *Guide: Choose **None** to run without LoRA, **auto** to use the most recent LoRA, or type/paste the exact filename from Step 2 (Available LoRAs).*
LORA_FILENAME = "None" #@param ["None", "auto"] {allow-input: true}
LORA_Trigger_Words = "" #@param {type:"string"}
LORA_STRENGTH = 1.0 #@param {type:"slider", min:0.0, max:2.0, step:0.05}

# --- Generation Settings ---
POSITIVE_PROMPT = "A beautiful futuristic city at sunset." #@param {type:"string"}
NEGATIVE_PROMPT = "blurry, low quality, deformed, artifacts, bad anatomy" #@param {type:"string"}
BATCH_SIZE = 1 #@param {type:"slider", min:1, max:4, step:1}
WIDTH = 640 #@param {type:"slider", min:512, max:2048, step:64}
HEIGHT = 640 #@param {type:"slider", min:512, max:2048, step:64}
PROMPT = POSITIVE_PROMPT if LORA_FILENAME == "None" else LORA_Trigger_Words + POSITIVE_PROMPT

# --- Advanced Sampler Settings ---
STEPS = 9 #@param {type:"slider", min:1, max:20, step:1}
CFG = 1.0 #@param {type:"number"}
SAMPLER_NAME = "euler" #@param ["euler", "euler_ancestral", "heun", "dpm_2", "dpm_2_ancestral", "lms", "dpm_fast", "dpm_adaptive", "dpmpp_2s_ancestral", "dpmpp_sde", "dpmpp_sde_gpu", "dpmpp_2m", "dpmpp_2m_sde", "dpmpp_2m_sde_gpu", "dpmpp_3m_sde", "dpmpp_3m_sde_gpu", "ddpm", "lcm", "ddim", "uni_pc", "uni_pc_bh2", "res_multistep"] {allow-input: true}
SCHEDULER = "beta" #@param ["normal", "karras", "exponential", "sgm_uniform", "simple", "ddim_uniform", "beta"] {allow-input: true}
AURA_SHIFT = 3.0 #@param {type:"slider", min:1.0, max:10.0, step:0.5}
SEED = 0 #@param {type:"integer"}

# --- Direct Download (No ZIP) ---
AUTO_DOWNLOAD = False #@param {type:"boolean"}

import sys
import os
import json
import time
import random
import subprocess
import urllib.request
from google.colab import files
from IPython.display import display, Image as IPImage

WORKSPACE = "/content/ComfyUI"
os.chdir(WORKSPACE)

# Fixed GGUF Model Filename
UNET_FILENAME = "z-image-turbo-Q8_0.gguf"

# Auto-detect LoRA if set to 'auto'
if LORA_FILENAME.lower() == "auto":
    if 'ACTIVE_LORA_FILENAME' in globals() and ACTIVE_LORA_FILENAME:
      LORA_FILENAME = next(iter(recent_lora))

    else:
        lora_dir = os.path.join(WORKSPACE, "models/loras")
        available = [f for f in os.listdir(lora_dir) if f.endswith(('.safetensors', '.pt', '.ckpt'))] if os.path.exists(lora_dir) else []
        LORA_FILENAME = available[0] if available else "None"

if SEED == 0:
    SEED = random.randint(1, 1125899906842624)

print("🔌 [@CoinNoin] Checking ComfyUI Server Status...")
def start_server():
    req = urllib.request.Request("http://127.0.0.1:8188")
    try:
        urllib.request.urlopen(req)
        print("   🟢 Server is already running.")
    except:
        print("   🚀 Starting ComfyUI Server in background...")
        subprocess.Popen([sys.executable, "main.py"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        while True:
            try:
                urllib.request.urlopen(req)
                print("   🟢 Server is now up and ready!")
                break
            except:
                time.sleep(2)

start_server()

# Brand visibility prefix in filename
BRAND_PREFIX = "@CoinNoin_ZImage"

prompt_workflow = {
    "9": {"inputs": {"filename_prefix": BRAND_PREFIX, "images": ["43", 0]}, "class_type": "SaveImage"},
    "39": {"inputs": {"clip_name": "qwen_3_4b.safetensors", "type": "lumina2", "device": "default"}, "class_type": "CLIPLoader"},
    "40": {"inputs": {"vae_name": "ae.safetensors"}, "class_type": "VAELoader"},
    "41": {"inputs": {"width": WIDTH, "height": HEIGHT, "batch_size": BATCH_SIZE}, "class_type": "EmptySD3LatentImage"},
    "42": {"inputs": {"text": NEGATIVE_PROMPT, "clip": ["39", 0]}, "class_type": "CLIPTextEncode"},
    "43": {"inputs": {"samples": ["44", 0], "vae": ["40", 0]}, "class_type": "VAEDecode"},
    "44": {"inputs": {"seed": SEED, "steps": STEPS, "cfg": CFG, "sampler_name": SAMPLER_NAME, "scheduler": SCHEDULER, "denoise": 1, "model": ["47", 0], "positive": ["45", 0], "negative": ["42", 0], "latent_image": ["41", 0]}, "class_type": "KSampler"},
    "45": {"inputs": {"text": PROMPT, "clip": ["39", 0]}, "class_type": "CLIPTextEncode"},
    "47": {"inputs": {"shift": AURA_SHIFT, "model": ["48", 0]}, "class_type": "ModelSamplingAuraFlow"},
    "48": {"inputs": {"unet_name": UNET_FILENAME}, "class_type": "UnetLoaderGGUF"}
}

# Attach LoRA only if specified and exists
if LORA_FILENAME.strip() and LORA_FILENAME.lower() != "none":
    lora_path = os.path.join(WORKSPACE, "models/loras", LORA_FILENAME)
    if os.path.exists(lora_path):
        prompt_workflow["50"] = {
            "inputs": {
                "lora_name": LORA_FILENAME,
                "strength_model": LORA_STRENGTH,
                "model": ["48", 0]
            },
            "class_type": "LoraLoaderModelOnly"
        }
        prompt_workflow["47"]["inputs"]["model"] = ["50", 0]
        print(f"✨ [@CoinNoin] LoRA Applied: {LORA_FILENAME} (Strength: {LORA_STRENGTH})")
    else:
        print(f"⚠️ LoRA '{LORA_FILENAME}' not found in models/loras. Running without LoRA.")
else:
    print("ℹ️ [@CoinNoin] Running generation without LoRA.")

p = {"prompt": prompt_workflow}
data = json.dumps(p).encode('utf-8')
req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=data)

print(f"\033[94m➜ [@CoinNoin] Details | Res: {WIDTH}x{HEIGHT} | Seed: {SEED} | Steps: {STEPS}\033[0m")
print("📥 Submitting workflow to API...")
try:
    response = urllib.request.urlopen(req)
    prompt_id = json.loads(response.read())['prompt_id']
except Exception as e:
    print(f"❌ API Error: {e}")
    raise

print("✨ Processing and Sampling (Check ComfyUI server logs if stuck)...")
while True:
    try:
        history_req = urllib.request.Request(f"http://127.0.0.1:8188/history/{prompt_id}")
        history_res = urllib.request.urlopen(history_req)
        history_data = json.loads(history_res.read())
        if prompt_id in history_data:
            outputs = history_data[prompt_id]['outputs']
            break
    except:
        pass
    time.sleep(1)

print("🖼️ Decoding Final Masterpiece...")
generated_images = []
for node_id, node_output in outputs.items():
    if 'images' in node_output:
        for image in node_output['images']:
            filename = image['filename']
            img_path = os.path.join(WORKSPACE, "output", filename)
            generated_images.append(img_path)
            print(f"\033[92m✓ [@CoinNoin] Saved: {filename}\033[0m")
            display(IPImage(filename=img_path))

# Direct download trigger
if AUTO_DOWNLOAD and generated_images:
    print("\n📥 [@CoinNoin] Download Started...")
    for img_file in generated_images:
        files.download(img_file)